# Multimodal Crop Health Monitoring — Setup & Training

**Author:** Khang Phan  
**Purpose:** Download datasets, explore PlantVillage images, and train the disease classifier.

This notebook:
1. Checks that all dependencies are installed
2. Downloads all PlantVillage datasets and merges them
3. Explores the image dataset
4. Trains a ResNet-18 disease classifier
5. Saves the model to `results/disease_model.pth`

## 1. Environment Check

In [ ]:
import importlib.metadata, sys

# Maps pip package name → import name (for display)
packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "xgboost": "xgboost",
    "torch": "torch",
    "torchvision": "torchvision",
    "opencv-python": "cv2",
    "streamlit": "streamlit",
    "seaborn": "seaborn",
    "pillow": "PIL",
    "kaggle": "kaggle",
}

all_ok = True
for pip_name, import_name in packages.items():
    try:
        version = importlib.metadata.version(pip_name)
        print(f"  [OK]      {pip_name:<20} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  [MISSING] {pip_name:<20} → pip install {pip_name}")
        all_ok = False

print(f"\nPython {sys.version}")
print("\nEnvironment ready!" if all_ok else "\nInstall missing packages, then re-run.")

## 2. PlantVillage Dataset — Download

Clones the PlantVillage dataset from `gabrieldgf4/PlantVillage-Dataset` on GitHub into `data/raw/PlantVillage_combined/`. No Kaggle credentials needed.

In [ ]:
import os, json, shutil, subprocess
from pathlib import Path

# ── Setup paths ─────────────────────────────────────────────────────────
DATA_RAW = Path("../data/raw")
COMBINED = DATA_RAW / "PlantVillage_combined"

# Wipe and rebuild clean
if COMBINED.exists():
    shutil.rmtree(COMBINED)
    print("Cleared existing PlantVillage_combined")
COMBINED.mkdir(parents=True, exist_ok=True)

# ── Clone gabrieldgf4/PlantVillage-Dataset ───────────────────────────────
print("Cloning gabrieldgf4/PlantVillage-Dataset...")
gh_dir = DATA_RAW / "gabrieldgf4"
if gh_dir.exists():
    shutil.rmtree(gh_dir)

result = subprocess.run(
    ["git", "clone", "--depth=1",
     "https://github.com/gabrieldgf4/PlantVillage-Dataset.git", str(gh_dir)],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

# Find the color image folder
candidates = [
    gh_dir / "dataset" / "color",
    gh_dir / "color",
    gh_dir / "dataset",
]
src_dir = next((p for p in candidates if p.exists() and any(p.iterdir())), None)
if src_dir is None:
    raise FileNotFoundError(f"Could not find image folder. Contents: {list(gh_dir.rglob('*'))[:10]}")
print(f"Source folder: {src_dir}")

# Copy into combined (skip .git and hidden dirs)
copied, skipped = 0, 0
for class_dir in sorted(src_dir.iterdir()):
    if not class_dir.is_dir() or class_dir.name.startswith("."):
        continue
    dest = COMBINED / class_dir.name
    dest.mkdir(exist_ok=True)
    for img in class_dir.glob("*"):
        if img.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            shutil.copy2(img, dest / img.name)
            copied += 1

# ── Summary ─────────────────────────────────────────────────────────────
print(f"\n=== Dataset Summary ===")
class_folders = sorted([d for d in COMBINED.iterdir() if d.is_dir()])
total = sum(len(list(d.glob("*"))) for d in class_folders)
print(f"Classes : {len(class_folders)}")
print(f"Images  : {total:,}")
for d in class_folders:
    n = len(list(d.glob("*")))
    print(f"  {d.name:<50} {n:>5} images")

## 3. PlantVillage Image Dataset — Exploration

Visualize class distribution across the merged dataset.

In [ ]:
import pathlib, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

DATA_RAW = pathlib.Path("../data/raw")
candidates = [
    DATA_RAW / "PlantVillage_combined",
    DATA_RAW / "abdallahalidev" / "plantvillage dataset" / "color",
    DATA_RAW / "abdallahalidev" / "color",
    DATA_RAW / "emmarex" / "PlantVillage",
    DATA_RAW / "PlantVillage",
    DATA_RAW / "color",
]
pv_dir = next((p for p in candidates if p.exists() and any(p.iterdir())), None)

if pv_dir:
    class_counts = {d.name: len(list(d.glob("*")))
                    for d in sorted(pv_dir.iterdir()) if d.is_dir()}
    print(f"Dataset: {pv_dir}")
    print(f"Classes: {len(class_counts)} | Images: {sum(class_counts.values()):,}")
else:
    print("PlantVillage not found — run Section 2 first.")
    class_counts = {
        "Tomato_healthy": 1591, "Tomato_Early_blight": 1000, "Tomato_Late_blight": 1909,
        "Tomato_Leaf_Mold": 952, "Tomato_Septoria_leaf_spot": 1771,
        "Potato_healthy": 152, "Potato_Early_blight": 1000, "Potato_Late_blight": 1000,
        "Pepper_healthy": 1478, "Pepper_Bacterial_spot": 997,
    }

top = dict(sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:15])
colors = ["#4CAF50" if "healthy" in k.lower() else "#F44336" for k in top]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(list(top.keys()), list(top.values()), color=colors, alpha=0.85, edgecolor="white")
ax.set_xlabel("Number of Images")
ax.set_title("PlantVillage — Top 15 Classes by Image Count
(green = healthy, red = disease)",
             fontweight="bold")
ax.bar_label(bars, padding=3, fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS / "eda_plantvillage.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Plot saved to results/eda_plantvillage.png")

## 4. Sample Image Grid (PlantVillage)

In [ ]:
import random, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

pv_dir = Path("../data/raw/PlantVillage")
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle("Sample Leaf Images — PlantVillage Dataset", fontsize=13, fontweight="bold")

all_images = list(pv_dir.rglob("*.jpg")) + list(pv_dir.rglob("*.JPG")) if pv_dir.exists() else []

for i, ax in enumerate(axes.flat):
    if all_images:
        img_path = random.choice(all_images)
        img = Image.open(img_path).resize((224, 224))
        ax.imshow(img)
        label = img_path.parent.name.replace("_", "\n")
    else:
        # Synthetic placeholder: colored square with label
        color = np.random.uniform(0.1, 0.9, (224, 224, 3))
        color[:, :, 1] = color[:, :, 1] * 0.6 + 0.2  # greenish tint
        ax.imshow(color)
        labels = ["Tomato_healthy", "Tomato_Early_blight", "Potato_Late_blight",
                  "Corn_healthy", "Apple_scab", "Grape_Black_rot",
                  "Pepper_healthy", "Corn_Common_rust", "Potato_healthy"]
        label = labels[i].replace("_", "\n")
    ax.set_title(label, fontsize=7)
    ax.axis("off")

plt.tight_layout()
plt.savefig("../results/sample_images.png", dpi=100, bbox_inches="tight")
plt.show()
note = "(synthetic placeholders)" if not all_images else "(real PlantVillage images)"
print(f"Image grid {note} saved to results/sample_images.png")

---

## 5. Multimodal Disease Classifier — Data Loading

Loads PlantVillage images and wraps them with a **WeatherAugmentedDataset** that
generates synthetic weather (temperature, humidity, days since rain) biased by disease type.

This gives the model a genuine training signal: fungal disease images are paired with
humid/warm weather; viral/insect images with hot/dry weather. At inference, the user's
real weather slider values are passed in.

> **Tip:** Enable GPU in Colab → Runtime → Change runtime type → T4 GPU.

In [ ]:
import sys, torch, shutil
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset, random_split
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path("../src").resolve()))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

DATA_RAW = Path("../data/raw")
candidates = [
    DATA_RAW / "PlantVillage_combined",
    DATA_RAW / "abdallahalidev" / "plantvillage dataset" / "color",
    DATA_RAW / "abdallahalidev" / "color",
    DATA_RAW / "emmarex" / "PlantVillage",
    DATA_RAW / "PlantVillage",
    DATA_RAW / "color",
]
PV_DIR = next((p for p in candidates if p.exists() and any(p.iterdir())), None)
if PV_DIR is None:
    raise FileNotFoundError("No PlantVillage folder found. Run Section 2 first.")
print(f"Using dataset: {PV_DIR}")

# Remove hidden/system directories that ImageFolder would mistake for classes
for d in PV_DIR.iterdir():
    if d.is_dir() and d.name.startswith("."):
        print(f"Removing hidden directory: {d.name}")
        shutil.rmtree(d)

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

base_dataset = datasets.ImageFolder(PV_DIR, transform=train_transform)
CLASS_NAMES  = base_dataset.classes
NUM_CLASSES  = len(CLASS_NAMES)

# Weather bias per disease category
WEATHER_BIAS = {
    "fungal":    (23, 4,  82, 8,  1.5, 1.0),
    "bacterial": (26, 4,  76, 8,  2.0, 1.5),
    "viral":     (32, 4,  38, 12, 9.0, 3.0),
    "mite":      (35, 4,  30, 10, 10.0, 3.0),
    "healthy":   (22, 5,  60, 12, 4.0, 2.0),
}

FUNGAL    = {"Tomato_Early_blight","Tomato_Late_blight","Tomato_Leaf_Mold",
             "Tomato_Septoria_leaf_spot","Tomato__Target_Spot",
             "Potato___Early_blight","Potato___Late_blight"}
BACTERIAL = {"Tomato_Bacterial_spot","Pepper__bell___Bacterial_spot"}
VIRAL     = {"Tomato__Tomato_YellowLeaf__Curl_Virus","Tomato__Tomato_mosaic_virus"}
MITE      = {"Tomato_Spider_mites_Two_spotted_spider_mite"}

def _weather_bias(class_name):
    if class_name in FUNGAL:    return WEATHER_BIAS["fungal"]
    if class_name in BACTERIAL: return WEATHER_BIAS["bacterial"]
    if class_name in VIRAL:     return WEATHER_BIAS["viral"]
    if class_name in MITE:      return WEATHER_BIAS["mite"]
    return WEATHER_BIAS["healthy"]

class WeatherAugmentedDataset(Dataset):
    """Wraps ImageFolder; adds disease-biased synthetic weather per sample."""
    def __init__(self, base, class_names):
        self.base = base
        self.class_names = class_names
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        image, label = self.base[idx]
        mt, st, mh, sh, mr, sr = _weather_bias(self.class_names[label])
        temp     = float(np.clip(np.random.normal(mt, st), -10, 50))
        humidity = float(np.clip(np.random.normal(mh, sh),   0, 100))
        rain     = float(np.clip(np.random.normal(mr, sr),   0,  30))
        weather  = torch.tensor([temp, humidity, rain], dtype=torch.float32)
        return image, weather, label

full_ds = WeatherAugmentedDataset(base_dataset, CLASS_NAMES)
n_val   = int(0.2 * len(full_ds))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val],
                                generator=torch.Generator().manual_seed(42))
val_ds.dataset.base.transform = val_transform

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Classes : {NUM_CLASSES}")
print(f"Train   : {n_train:,} images")
print(f"Val     : {n_val:,} images")


## 6. Build ResNet-18 Disease Classifier

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}  /  {total:,} total")
print(f"Model ready on   : {DEVICE}")

## 7. Train ResNet-18 Disease Classifier

In [ ]:
EPOCHS = 10
history = {"train_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    # Training
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, weather, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc  = correct / total

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, weather, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            preds  = model(images).argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)
    val_acc = val_correct / val_total

    scheduler.step()
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    print(f"Epoch {epoch}/{EPOCHS}  loss={train_loss:.4f}  "
          f"train_acc={train_acc:.3f}  val_acc={val_acc:.3f}")

## 8. Training Curves & Save Model

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("ResNet-18 Disease Classifier — Training Results", fontweight="bold")

axes[0].plot(epochs_range, history["train_loss"], marker="o", color="#F44336", label="Train Loss")
axes[0].set_title("Loss per Epoch"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], marker="o", color="#2196F3", label="Train Acc")
axes[1].plot(epochs_range, history["val_acc"],   marker="s", color="#4CAF50", label="Val Acc")
axes[1].set_title("Accuracy per Epoch"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS / "disease_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

torch.save({
    "model_state": model.state_dict(),
    "classes":     CLASS_NAMES,
    "model_type":  "resnet18",
}, RESULTS / "disease_model.pth")
print(f"Model saved → results/disease_model.pth")
print(f"Final val accuracy: {history['val_acc'][-1]:.3f}")

---

## 9. XGBoost Environmental Risk Model

Trains XGBoost to predict disease risk score from:
- `temperature`, `humidity`, `days_since_rain` (weather)
- `disease_class` (from ResNet output)
- `confidence` (ResNet softmax confidence)

**Training data:** generated by sampling weather from disease-specific distributions
(the same ones used to train the CNN) and computing risk scores via the agronomic
rules in `fusion.py`. XGBoost learns a smooth, non-linear version of those rules —
and can be updated with real field data later.

This replaces the hand-coded `if humidity > 60` logic in `fusion.py` with a learned model.

In [ ]:
import sys, numpy as np, pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))
from fusion import assess_risk, DISEASE_CONDITIONS
from recommendations import RECOMMENDATIONS

np.random.seed(42)
N_PER_CLASS = 500   # samples per disease class

# Weather sampling ranges per disease type (mirrors WeatherAugmentedDataset)
WEATHER_BIAS = {
    "fungal":    dict(temp=(23,4),  humidity=(82,8),  rain=(1.5,1.0)),
    "bacterial": dict(temp=(26,4),  humidity=(76,8),  rain=(2.0,1.5)),
    "viral":     dict(temp=(32,4),  humidity=(38,12), rain=(9.0,3.0)),
    "mite":      dict(temp=(35,4),  humidity=(30,10), rain=(10.0,3.0)),
    "healthy":   dict(temp=(22,5),  humidity=(60,12), rain=(4.0,2.0)),
}

FUNGAL    = {c for dtype, info in DISEASE_CONDITIONS.items() if dtype == "fungal"    for c in info["classes"]}
BACTERIAL = {c for dtype, info in DISEASE_CONDITIONS.items() if dtype == "bacterial" for c in info["classes"]}
VIRAL     = {c for dtype, info in DISEASE_CONDITIONS.items() if dtype == "viral"     for c in info["classes"]}
MITE      = {c for dtype, info in DISEASE_CONDITIONS.items() if dtype == "mites"     for c in info["classes"]}

def disease_type(cls):
    if cls in FUNGAL:    return "fungal"
    if cls in BACTERIAL: return "bacterial"
    if cls in VIRAL:     return "viral"
    if cls in MITE:      return "mite"
    return "healthy"

ALL_CLASSES = list(RECOMMENDATIONS.keys())

rows = []
for cls in ALL_CLASSES:
    dtype   = disease_type(cls)
    bias    = WEATHER_BIAS[dtype]
    severity = RECOMMENDATIONS[cls]["severity"]

    for _ in range(N_PER_CLASS):
        temp     = float(np.clip(np.random.normal(*bias["temp"]),     -10, 50))
        humidity = float(np.clip(np.random.normal(*bias["humidity"]),   0, 100))
        rain     = float(np.clip(np.random.normal(*bias["rain"]),       0,  30))
        confidence = float(np.clip(np.random.beta(8, 2), 0.4, 1.0))   # realistic softmax dist

        risk = assess_risk(cls, confidence, severity, temp, humidity, rain)
        rows.append({
            "disease_class": cls,
            "temperature":   temp,
            "humidity":      humidity,
            "days_since_rain": rain,
            "confidence":    confidence,
            "risk_score":    risk["risk_score"],
        })

df = pd.DataFrame(rows)
print(f"Dataset: {df.shape}  |  classes: {df['disease_class'].nunique()}")
print(df.groupby("disease_class")["risk_score"].mean().round(3).to_string())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Generated Training Data — Feature vs Risk Score", fontweight="bold")

for ax, feat in zip(axes, ["temperature", "humidity", "days_since_rain"]):
    ax.scatter(df[feat], df["risk_score"], alpha=0.05, s=5, color="#2196F3")
    ax.set_xlabel(feat)
    ax.set_ylabel("risk_score")

plt.tight_layout()
plt.savefig(Path("../results") / "xgb_training_data.png", dpi=120, bbox_inches="tight")
plt.show()

print("Risk score distribution:")
print(df["risk_score"].describe().round(3))


## 10. Train XGBoost Risk Model

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

le = LabelEncoder()
df["class_enc"] = le.fit_transform(df["disease_class"])

FEATURES = ["temperature", "humidity", "days_since_rain", "confidence", "class_enc"]
X = df[FEATURES].values
y = df["risk_score"].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_xgb = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

y_pred = model_xgb.predict(X_val)
print(f"\nVal MAE : {mean_absolute_error(y_val, y_pred):.4f}")
print(f"Val R²  : {r2_score(y_val, y_pred):.4f}")


In [ ]:
import pickle, matplotlib.pyplot as plt
from pathlib import Path

RESULTS = Path("../results")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("XGBoost Risk Model — Evaluation", fontweight="bold")

axes[0].scatter(y_val, y_pred, alpha=0.2, s=8, color="#2196F3")
lims = [0, 1]
axes[0].plot(lims, lims, "r--", linewidth=1)
axes[0].set_xlabel("Actual risk score")
axes[0].set_ylabel("Predicted risk score")
axes[0].set_title(f"Predicted vs Actual  (R²={r2_score(y_val, y_pred):.3f})")

feat_labels = ["temperature", "humidity", "days_since_rain", "confidence", "disease_class"]
axes[1].barh(feat_labels, model_xgb.feature_importances_, color="#4CAF50", alpha=0.85, edgecolor="white")
axes[1].set_title("Feature Importance")

plt.tight_layout()
plt.savefig(RESULTS / "xgb_risk_evaluation.png", dpi=120, bbox_inches="tight")
plt.show()

# Save
with open(RESULTS / "risk_model.pkl", "wb") as f:
    pickle.dump({"model": model_xgb, "label_encoder": le, "features": FEATURES}, f)
print("Saved to results/risk_model.pkl")

# Sanity check
print("\nSanity checks:")
for cls, temp, hum, rain, conf in [
    ("Tomato_Late_blight",  25, 90, 1, 0.95),   # expect Critical
    ("Tomato_Early_blight", 20, 55, 5, 0.80),   # expect Moderate
    ("Tomato_healthy",      22, 60, 4, 0.99),   # expect Low
]:
    enc = le.transform([cls])[0]
    score = model_xgb.predict([[temp, hum, rain, conf, enc]])[0]
    print(f"  {cls:<45} score={score:.3f}")
